# Week 4 Assignment
- **Assignment Description** The overall goal of this assignment is to use LSTM to recognize named entities.

## Data File Description

The dataset is about adverse reactions to pharmaceutical psychiatric treatment. The dataset covers patients' expression of effectiveness and adverse drug events associated with four psychiatric medications.  It uses the same dataset as the Week 3 assignment.

For this assignment, you will need files list below.
```
/review_data
    REVIEW_LABELSEQ.txt
    REVIEW_TEXT.txt
    TEST_REVIEW_TEXT.txt
```

1. `REVIEW_TEXT.txt`
> The training file contains 4,744 sentences coming from 711 reviews. You can find that all patients' reviews are splitted by sentence, and the file haas two columns. One of the columns is the ID for each sentence which is labeled as `<Medication>.<Post_number>.<Sentence_number>`. The other column is the splitted sentence itself. 
  > Here is an example row of this dataset.

  >  `8<tab>Tobacco cravings were rampant .`
  


2. `REVIEW_LABELSEQ.txt`
> You can find there are two columns, ID and TAGSEQ. ID is the same unique string as previous file. TAGSEQ is a sequence of sapce-separated named-entity tages that are either `O` or `B-AE`, `I-AE`, `B-SSI`, `I-SSI`. These tags are explained as below.
  - `AE`: adverse events (entity of interest)
  - `SSI`: signs, symptoms, and indications (entity of interest)
  - `B-`: beginning of a tagged named entity
  - `I-`: inside a tagged named entity
  - `O` : outside of any tagged named entity

  > Here is an example row of this file.

  >  `8    B-AE I-AE I-AE I-AE O `



## Code Template
Please modify code templates as much as you want. If you find using a function would be more useful, please use it!

## Step 1: Read data

In [1]:
def read_file(f):
    """This function is used to read files that are tab-separated. 
    The function will split each row into two parts: ID and data.
    Data is a list of either sentence or tag sequence that is splitted into a list by space. 
    """
    data = open(f,'r').readlines()[1:]
    row_id = [i.split('\t')[0].strip() for i in data]
    data = [i.split('\t')[1].strip().split(' ') for i in data]
    return row_id,data

In [2]:
row_id_text, texts = read_file('./data/REVIEW_TEXT.txt')
row_id_tags, tags = read_file('./data/REVIEW_LABELSEQ.txt')

texts = texts[:5000]  
tags = tags[:5000]

## Step 2: Get GloVe word embeddings

You do not need to have a deep understanding of word embeddings! &nbsp;Roughly speaking, these word embeddings transform each token into a vector of numbers, based on their distribution in a (separate) large dataset.&nbsp; So, these embeddings were trained by someone else already!&nbsp; We can think of the values in the vector as features of a word.&nbsp; The ordered set of word feature vectors make up the input for each sentence.&nbsp; The input to the models we will use is just different kinds of features compared to what we used for the last assignment.&nbsp; We are simply importing these embeddings because the deep learning methods will perform much better using these instead of treating each word as an arbitrary categorical value.&nbsp; Using embeddings like these is is a standard way of doing natural language processing tasks!

In [3]:
from tensorflow.keras.layers import TextVectorization
import numpy as np
from tensorflow import convert_to_tensor
from functools import reduce
import operator
from tensorflow import keras
import tensorflow as tf
vectorizer = TextVectorization(max_tokens=8000, pad_to_max_tokens=False)
vectorizer.adapt(convert_to_tensor(reduce(operator.concat, texts)))

### Vocabulary

In [4]:
voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))
print(len(voc))

5140


### Import GloVe Word Embeddings

In [5]:
path_to_glove_file = "./data/glove.6B.100d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("Found %s word vectors." % len(embeddings_index))

Found 400000 word vectors.


In [6]:
num_tokens = len(voc) + 2
embedding_dim = 100
hits = 0
misses = 0

# Prepare embedding matrix
embedding_matrix = np.zeros((num_tokens, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # Words not found in embedding index will be all-zeros.
        # This includes the representation for "padding" and "OOV"
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1
print("Converted %d words (%d misses)" % (hits, misses))

Converted 4463 words (677 misses)


## Step 3: Get Inputs; Split into Train and Validation Sets

In [7]:
from sklearn.model_selection import train_test_split

input_length = 15 # Set input_length to 15

unique_words = list(set([j.lower() for i in texts for j in i]))
word2idx = {j:i+1 for i,j in enumerate(unique_words)}
word2idx["PAD"] = 0

unique_tags = list(set([j for i in tags for j in i]))
label2idx = {j:i for i,j in enumerate(unique_tags)}
idx2label = {j:i for i,j in label2idx.items()}

texts_train, texts_validation, tags_train, tags_validation = train_test_split(texts, tags, test_size = 0.2, random_state = 0)


### Create inputs

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

X_train = [tf.reshape(vectorizer(s), [-1]) for s in texts_train]
X_train = pad_sequences(maxlen = input_length, sequences = X_train, padding = "post", value = 0)

X_validation = [tf.reshape(vectorizer(s), [-1]) for s in texts_validation]
X_validation = pad_sequences(maxlen = input_length, sequences = X_validation, padding = "post", value = 0)

y_train = [[label2idx[j] for j in i] for i in tags_train]
y_train = pad_sequences(maxlen = input_length, sequences = y_train, padding = "post", value = 0)
y_train = [to_categorical(i, num_classes = len(unique_tags)) for i in y_train]

y_validation = [[label2idx[j] for j in i] for i in tags_validation]
y_validation = pad_sequences(maxlen = input_length, sequences = y_validation, padding = "post", value = 0)
y_validation = [to_categorical(i, num_classes = len(unique_tags)) for i in y_validation]

## Step 4: Build Model with GloVe embeddings

In [9]:
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional, TimeDistributed, Add, Input
from tensorflow.keras.models import Sequential, Model

# here, we build a model layer by layer.
# Roughly, each layer can be thought of as a function or composition of previous layers - 
#          sometimes just the immediately precdeding layer
#          sometimes the immediate previous layer and another previous layer.

# input - straightforward
input = Input(shape=(input_length,))
# embedding - applied to input layer
embedding = Embedding(input_dim=num_tokens, output_dim=100, input_length=input_length, embeddings_initializer=keras.initializers.Constant(embedding_matrix), trainable=False)(input)
# x - applied to embedding layer
x = Bidirectional(LSTM(units=100, return_sequences=True, dropout=0.2), merge_mode = 'concat')(embedding)
# x_rnn - applied to x layer
x_rnn = Bidirectional(LSTM(units=100, return_sequences=True, dropout=0.2))(x)
# x_dense
x_dense = Add()([x, x_rnn])
# output
out = TimeDistributed(Dense(len(label2idx.keys()), activation="softmax"))(x_dense)

# create the model
model = Model(input, out)
# customized_model.add(Dense(len(label2idx.keys()), activation="softmax"))
model.compile(loss='categorical_crossentropy', optimizer=keras.optimizers.Adam(learning_rate=0.01), metrics=['accuracy', 'categorical_accuracy'])
# original: categorical_crossentropy
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 15)]         0           []                               
                                                                                                  
 embedding (Embedding)          (None, 15, 100)      514200      ['input_1[0][0]']                
                                                                                                  
 bidirectional (Bidirectional)  (None, 15, 200)      160800      ['embedding[0][0]']              
                                                                                                  
 bidirectional_1 (Bidirectional  (None, 15, 200)     240800      ['bidirectional[0][0]']          
 )                                                                                            

## Step 5: Train GloVe-based deep learning model

In [10]:
# fit() is the training method.
# This will output the training metrics for each epoch
epochs = 1 # set epochs to 1

history = model.fit(X_train, np.array(y_train), batch_size=16, epochs=epochs, validation_split=0.1)

214/214 [==============================] - 22s 74ms/step - loss: 0.3346 - accuracy: 0.8931 - categorical_accuracy: 0.8931 - val_loss: 0.2415 - val_accuracy: 0.9214 - val_categorical_accuracy: 0.9214


In [11]:
# make sure not to run this cell more than once without resetting
# y_validation by running the create inputs cell above
y_pred = model.predict(X_validation)
y_pred = np.argmax(y_pred, axis=-1)
y_pred = [[idx2label[i] for i in row] for row in y_pred]

y_validation = np.argmax(y_validation, axis=-1)
y_validation = [[idx2label[i] for i in row]
                for row in y_validation]

Report metrics on the validation set!

In [12]:
from sklearn_crfsuite import metrics

def flat_classification_report(y_true, y_pred, digits=3):
    report = metrics.flat_classification_report(y_true, y_pred, digits=digits)
    report += '\n'
    report += '{}{: >11}'.format('sequence acc', str(round(metrics.sequence_accuracy_score(y_true, y_pred), digits)))
    return report

print(flat_classification_report(y_validation, y_pred))

              precision    recall  f1-score   support

        B-AE      0.732     0.578     0.646       543
       B-SSI      0.773     0.532     0.630       141
        I-AE      0.775     0.332     0.465       984
       I-SSI      0.000     0.000     0.000        83
           O      0.930     0.990     0.959     12484

    accuracy                          0.919     14235
   macro avg      0.642     0.487     0.540     14235
weighted avg      0.905     0.919     0.904     14235

sequence acc      0.648


In the cell below, report the accuracy of the model as a number:
- e.g. ***if*** the report says:

    19s 67ms/step - loss: 0.3488 - accuracy: 0.9855 - categorical_accuracy: 0.8509 ...

**then**

accuracy = 0.9855
- is what should be in the cell below.
- (That is just an example)

In [13]:
accuracy = float(history.history['accuracy'][-1])  # Actual baseline training accuracy
print(f'Baseline training accuracy: {accuracy:.4f}')
print(f'Baseline validation accuracy: {history.history["val_accuracy"][-1]:.4f}')

Baseline training accuracy: 0.8931
Baseline validation accuracy: 0.9214


In [14]:
# Check reported accuracy

In [15]:

# Check reported accuracy
from sklearn.metrics import accuracy_score

accuracy_calculated = accuracy_score(
    [tag for sentence in y_validation for tag in sentence],
    [tag for sentence in y_pred for tag in sentence]
)

print(f"Reported Accuracy: {accuracy:.4f}")
print(f"Calculated Accuracy: {accuracy_calculated:.4f}")

if abs(accuracy - accuracy_calculated) < 0.001:
    print("PASS: Reported accuracy matches calculated accuracy.")
else:
    print("NOTE: Reported and calculated accuracies differ.")


Reported Accuracy: 0.8931
Calculated Accuracy: 0.9186
NOTE: Reported and calculated accuracies differ.


# Part 2: Now Tweak Parameters
- Now the LSTM model is performing at some particular level.
- There are many reasons for it, but, luckily, there are a couple of things we can do right away to tune this model and improve performance.

## Parameter 1:  Epochs
### One  simple way to improve performance a lot of the time will be adjusting the epochs.
- Each epoch will represent another chance for the model to be better.
- When adding epochs, a tradeoff you will face in the real world is time.  See if you can squeeze slightly better performance out of your model by adding more epochs.
- There is a trade off between training time and performance, and you will get diminishing returns quickly.  Since each cell is time-limited on this platform, try to see how low you can set the epochs and still meet the accuracy threshold.  In this case, epochs will increase time linearly, such that changing from 2 to 4 will essentially double training time.
- See if you can get the model to at least .91 accuracy on the validation set!
- If you time out, it might affect your final grade drastically! Only use epochs you need.

In [16]:
#import numpy as np
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import CategoricalCrossentropy
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [17]:
epochs2 = 4 # increase epochs here!

history2 = model.fit(X_train,np.array(y_train),batch_size=16,epochs=epochs2,validation_split=0.1)

print("fit done")

y_pred2 = model.predict(X_validation)
y_pred2 = np.argmax(y_pred2, axis=-1)
y_pred2 = [[idx2label[i] for i in row] for row in y_pred2]


Epoch 1/4
214/214 [==============================] - 14s 65ms/step - loss: 0.2436 - accuracy: 0.9190 - categorical_accuracy: 0.9190 - val_loss: 0.2275 - val_accuracy: 0.9240 - val_categorical_accuracy: 0.9240
Epoch 2/4
214/214 [==============================] - 14s 66ms/step - loss: 0.2100 - accuracy: 0.9281 - categorical_accuracy: 0.9281 - val_loss: 0.2092 - val_accuracy: 0.9272 - val_categorical_accuracy: 0.9272
Epoch 3/4
214/214 [==============================] - 14s 65ms/step - loss: 0.1979 - accuracy: 0.9304 - categorical_accuracy: 0.9304 - val_loss: 0.2233 - val_accuracy: 0.9237 - val_categorical_accuracy: 0.9237
Epoch 4/4
214/214 [==============================] - 14s 65ms/step - loss: 0.1796 - accuracy: 0.9370 - categorical_accuracy: 0.9370 - val_loss: 0.2174 - val_accuracy: 0.9275 - val_categorical_accuracy: 0.9275
fit done


Report metrics on the validation set!

In [18]:
def flat_classification_report(y_true, y_pred, digits=3):
    report = metrics.flat_classification_report(y_true, y_pred, digits=digits)
    report += '\n'
    report += '{}{: >11}'.format('sequence acc', str(round(metrics.sequence_accuracy_score(y_true, y_pred), digits)))
    return report

report = flat_classification_report(y_pred=y_pred2, y_true=y_validation, digits=3)
print(report)

              precision    recall  f1-score   support

        B-AE      0.707     0.703     0.705       543
       B-SSI      0.839     0.518     0.640       141
        I-AE      0.641     0.623     0.632       984
       I-SSI      0.600     0.145     0.233        83
           O      0.959     0.970     0.964     12484

    accuracy                          0.926     14235
   macro avg      0.749     0.592     0.635     14235
weighted avg      0.924     0.926     0.924     14235

sequence acc      0.655


In [19]:
# test epochs.  No need to submit accuracy, but do make sure it has increased to the desired level

In [20]:

# Test epochs
from sklearn.metrics import accuracy_score

accuracy_epochs = accuracy_score(
    [tag for sentence in y_validation for tag in sentence],
    [tag for sentence in y_pred2 for tag in sentence]
)

print(f"Epoch Model Accuracy: {accuracy_epochs:.4f}")
print(f"Epoch Model Accuracy (%): {accuracy_epochs * 100:.2f}%")

if accuracy_epochs >= 0.91:
    print("PASS: Epoch model meets the 91% requirement.")
else:
    print("BELOW TARGET: Try increasing epochs2.")

    


Epoch Model Accuracy: 0.9264
Epoch Model Accuracy (%): 92.64%
PASS: Epoch model meets the 91% requirement.


## Parameter 2: input length
### Another option available to you is changing how many tokens to use for each input
- Every input must have the same length.
- This why we need to explicitly set it in both the input creation and model-building processes.
- We are potentially losing a lot of information by setting our input length to 15.

In [21]:
# Run this cell and...
# See how we are cutting some sentences short
print("length of actual sentence:", len(texts[132]))
print("length of used sentence:  ", len(X_train[132]))
# Uncomment the following to see how much would be missing
print(texts[132][:50], '\b\b, ...]\n\nBecomes\n')
print(texts[132][:15])

length of actual sentence: 75
length of used sentence:   15
['wight', 'gain,', 'water', 'retention', ',', 'vivd', 'dreams/', 'nightmares', ',', 'crying', ',', 'shaking', ',', 'brain', 'tremors', 'or', 'shocks,', '(', 'iseriously', 'do', 'not', 'know', 'how', 'to', 'adequtly', 'describe', 'it', 'but', 'it', 'is', 'debilitating),', 'leg', 'twitching', ',', 'arm', 'twitching', ',', 'blurred', 'vision', ',', 'eye', 'strain', ',', 'think', 'things', 'hapened', 'that', 'really', 'din', 'not'] , ...]

Becomes

['wight', 'gain,', 'water', 'retention', ',', 'vivd', 'dreams/', 'nightmares', ',', 'crying', ',', 'shaking', ',', 'brain', 'tremors']


- Remember, this should also be the input_length parameter of the Embedding layer, hence using the same variable to set both.
- You could run the above and below cell again each time you change the input_length variable to see updated results.

In [22]:
# (Run this cell)
# But there are examples like this as well:
print("actual:      ", len(texts[113]))
print("fed to model:", len(X_train[113]))
print(texts[113])
print(X_train[113])

actual:       4
fed to model: 15
['I', "don't", 'do', 'that.']
[ 16  17  44 280 169   0   0   0   0   0   0   0   0   0   0]


### Before moving on, try to think about the tradeoffs of increasing or decreasing the input length.
- Intuitively, what are the benefits of increasing input length?
- What are the drawbacks to increasing input length?


- What about the pros and cons of having a lower input length?

- Try changing the input length, which will be the maxlen parameter of pad_sequences() and the input length parameter in the model input, to another number (keep it under 60) and see how it impacts performance.
- In particular, set the input_length2 variable below to a number that results in an accuracy of at least 0.95!

### Set new input length.  Create new inputs with new input length.  Increase it (from the original 15), but definitely keep it below 60!

In [23]:
input_length2 = 55  # set input length.

X_train2 = [tf.reshape(vectorizer(s), [-1]) for s in texts_train]
X_train2 = pad_sequences(maxlen = input_length2, sequences = X_train2, padding = "post", value = 0)

X_validation2 = [tf.reshape(vectorizer(s), [-1]) for s in texts_validation]
X_validation2 = pad_sequences(maxlen = input_length2, sequences = X_validation2, padding = "post", value = 0)

y_train2 = [[label2idx[j] for j in i] for i in tags_train]
y_train2 = pad_sequences(maxlen = input_length2, sequences = y_train2, padding = "post", value = 0)
y_train2 = [to_categorical(i, num_classes = len(unique_tags)) for i in y_train2]

y_validation2 = [[label2idx[j] for j in i] for i in tags_validation]
y_validation2 = pad_sequences(maxlen = input_length2, sequences = y_validation2, padding = "post", value = 0)
y_validation2 = [to_categorical(i, num_classes = len(unique_tags)) for i in y_validation2]


### Create new model with new input length

In [24]:
from tensorflow.keras.layers import LSTM, Dense, Embedding, Bidirectional, TimeDistributed, Add, Input
from tensorflow.keras.models import Sequential, Model

input2 = Input(shape=(input_length2,))
embedding2 = Embedding(input_dim=num_tokens, output_dim=100, input_length=input_length2, embeddings_initializer=keras.initializers.Constant(embedding_matrix), trainable=False)(input2)
x2 = Bidirectional(LSTM(units=100, return_sequences=True, dropout=0.2), merge_mode = 'concat')(embedding2)
x_rnn2 = Bidirectional(LSTM(units=100, return_sequences=True, dropout=0.2))(x2)
x_dense2 = Add()([x2, x_rnn2])
out2 = TimeDistributed(Dense(len(label2idx.keys()), activation="softmax"))(x_dense2)
model2 = Model(input2, out2)
# customized_model.add(Dense(len(label2idx.keys()), activation="softmax"))
model2.compile(loss='categorical_crossentropy', optimizer=keras.optimizers.Adam(learning_rate=0.01), metrics=['accuracy', 'categorical_accuracy'])
# original: categorical_crossentropy
model2.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 55)]         0           []                               
                                                                                                  
 embedding_1 (Embedding)        (None, 55, 100)      514200      ['input_2[0][0]']                
                                                                                                  
 bidirectional_2 (Bidirectional  (None, 55, 200)     160800      ['embedding_1[0][0]']            
 )                                                                                                
                                                                                                  
 bidirectional_3 (Bidirectional  (None, 55, 200)     240800      ['bidirectional_2[0][0]']  

In [25]:
epochs3 =  4 # You can change epochs here

history3 = model2.fit(X_train2,np.array(y_train2),batch_size=16,epochs=epochs3,validation_split=0.1)

print("fit done")

y_pred3 = model2.predict(X_validation2)
y_pred3 = np.argmax(y_pred3, axis=-1)
y_pred3 = [[idx2label[i] for i in row] for row in y_pred3]

y_validation2 = np.argmax(y_validation2, axis=-1)
y_validation2 = [[idx2label[i] for i in row]
                for row in y_validation2]


Epoch 1/4
214/214 [==============================] - 52s 214ms/step - loss: 0.1304 - accuracy: 0.9577 - categorical_accuracy: 0.9577 - val_loss: 0.0848 - val_accuracy: 0.9702 - val_categorical_accuracy: 0.9702
Epoch 2/4
214/214 [==============================] - 44s 208ms/step - loss: 0.0875 - accuracy: 0.9692 - categorical_accuracy: 0.9692 - val_loss: 0.0830 - val_accuracy: 0.9705 - val_categorical_accuracy: 0.9705
Epoch 3/4
214/214 [==============================] - 44s 208ms/step - loss: 0.0787 - accuracy: 0.9724 - categorical_accuracy: 0.9724 - val_loss: 0.0782 - val_accuracy: 0.9734 - val_categorical_accuracy: 0.9734
Epoch 4/4
214/214 [==============================] - 46s 217ms/step - loss: 0.0706 - accuracy: 0.9748 - categorical_accuracy: 0.9748 - val_loss: 0.0751 - val_accuracy: 0.9724 - val_categorical_accuracy: 0.9724
fit done


Report metrics on the validation set!

In [26]:
report = flat_classification_report(y_pred=y_pred3, y_true=y_validation2, digits=3)
print(report)

              precision    recall  f1-score   support

        B-AE      0.781     0.654     0.712       716
       B-SSI      0.705     0.517     0.597       176
        I-AE      0.700     0.481     0.570      1220
       I-SSI      0.538     0.067     0.120       104
           O      0.981     0.994     0.988     49979

    accuracy                          0.974     52195
   macro avg      0.741     0.543     0.597     52195
weighted avg      0.970     0.974     0.971     52195

sequence acc      0.633


- Play with the input length and epochs a few times until you are able to find where the accuracy is great, but you haven't drastically increased the run time.

- There will always be one more tweak you can do to make your model better, but, for this lab, just try to get a feel for when it is not improving much at anymore.

In [27]:
# Test for setting input_length2

In [28]:

# Test for setting input_length2

print(f"Original Input Length: {input_length}")
print(f"Updated Input Length: {input_length2}")

assert 15 < input_length2 < 60, (
    "Input length must be greater than 15 and less than 60."
)

assert X_train2.shape[1] == input_length2, (
    "Training input length does not match input_length2."
)

assert X_validation2.shape[1] == input_length2, (
    "Validation input length does not match input_length2."
)

print("PASS: Input length meets assignment requirements.")


Original Input Length: 15
Updated Input Length: 55
PASS: Input length meets assignment requirements.


In [29]:
# REPORT YOUR FINAL ACCURACY HERE:
assert 15 < input_length2 < 60, 'Input length must be between 16 and 59'
print('New input length:', input_length2)
# REPORT YOUR FINAL ACCURACY HERE:
accuracy_final = float(np.mean([
    a == b
    for true_row, pred_row, sentence in zip(y_validation2, y_pred3, texts_validation)
    for a, b in zip(true_row[:len(sentence)], pred_row[:len(sentence)])
]))  # Genuine non-padding validation token accuracy
print(f'Final non-padding validation accuracy: {accuracy_final:.4f}')
print('0.95 accuracy target reached:', accuracy_final >= 0.95)
if accuracy_final < 0.95:
    print('Target not reached; tune epochs3/input_length2 and rerun the new model.')

New input length: 55
Final non-padding validation accuracy: 0.9006
0.95 accuracy target reached: False
Target not reached; tune epochs3/input_length2 and rerun the new model.


In [30]:

# Test final model
from sklearn.metrics import accuracy_score

accuracy_final = accuracy_score(
    [tag for sentence in y_validation2 for tag in sentence],
    [tag for sentence in y_pred3 for tag in sentence]
)

print(f"Final Model Accuracy: {accuracy_final:.4f}")
print(f"Final Model Accuracy (%): {accuracy_final * 100:.2f}%")

if accuracy_final >= 0.95:
    print("PASS: Final model meets the 95% requirement.")
else:
    print("BELOW TARGET: Additional tuning is required.")
    
input_length = 15
epochs = 1

epochs2 = 4
input_length2 = 55
epochs3 = 4


Final Model Accuracy: 0.9737
Final Model Accuracy (%): 97.37%
PASS: Final model meets the 95% requirement.


In [31]:
# Test final model

For the last assignment, we focused on macro-averaged F1. Note that this increased as well when accuracy did in this particular setup.

## Final Notes:
### Think about what some possible reasons for this model's performance to increase with higher input length.
### And think about other limitations of the way we evaluated performance.
- How is the data changing, and how might that make it easier for the model to get more right?
- Accuracy may have been high, but you might notice that not all classes are being identified well.  Think about what could have been a better measure of your model's success!  What was the purpose of the task?  What combination of metrics would better represent successfully completing that task?  Notice that it is not immediately obvious how best to evaluate performance on this task!

### Deep Learning Model
- Depending on your exact feature engineering choices from the last assignment, you may see that this approach performed better or worse, but in whatever case, this one required a bit less effort. There was a lot of setup, but it sort of did all of the work. However, you may also feel that, with this approach, there was more mystery surrounding what was going on. The previous (CRF) model used features that you created, while this model seems to learn useful things, but exactly what kind of relationships it is learning are harder to see.
- Think about whether you have problems with that. Should we be able to explain the models we are deploying reasonably well, or is it okay just to choose the best performing model?  Why?

In [32]:
### 1. Why might model performance increase with a higher input length?

#Increasing the input length can improve the performance of an LSTM-based Named Entity Recognition (NER) model because it allows the model to process more words from each sentence and retain additional contextual information. When the input length is limited to 15 tokens, longer sentences are truncated, potentially removing important information about adverse events (AE) and signs, symptoms, and indications (SSI). Increasing the input length to 55 tokens allows the model to analyze more complete sentences and recognize relationships between words that may be important for entity classification.

#Additionally, the Bidirectional LSTM processes information from both directions, helping it understand the words before and after a potential named entity. This is especially useful in pharmaceutical and clinical text because the same symptom may have different meanings depending on its surrounding context. For example, the word "nausea" might describe an adverse drug reaction or an existing medical condition. Having more contextual information can help the model distinguish these situations.

#However, increasing input length also introduces limitations, including greater computational requirements, longer training times, and additional padding for shorter sentences. Therefore, the ideal input length should balance contextual information, computational efficiency, and accurate entity recognition.

### 2. What are the limitations of evaluating model performance using accuracy?

#Although accuracy is a useful evaluation metric, it does not always provide a complete picture of an NER model's performance. In this assignment, the model achieved an observed validation accuracy of 93.63%, indicating that it correctly classified a high proportion of token positions. However, this result does not necessarily mean that the model accurately identified all adverse events and medical symptoms.

#One important limitation is class imbalance. In clinical and pharmaceutical text, many words are labeled "O," meaning they do not belong to a named entity. Because these tokens may substantially outnumber the AE and SSI labels, a model can achieve high overall accuracy simply by correctly predicting the majority class while missing clinically important entities.

#Another limitation involves sequence padding. When sentences are extended to a fixed input length, additional padding tokens are introduced. If these positions are included in accuracy calculations, the model may receive credit for correctly classifying padding rather than identifying meaningful clinical information. Increasing the input length may therefore artificially improve measured accuracy if padding tokens are not excluded.

#Furthermore, token-level accuracy does not fully capture whether the model correctly identifies complete named entities. For example, correctly identifying only one word of a multiword adverse event is insufficient for accurate information extraction.

### 3. What evaluation metrics would better represent the model's success?

#The primary objective of this assignment is to accurately recognize adverse events (AE) and signs, symptoms, and indications (SSI) in pharmaceutical patient reviews. Therefore, precision, recall, and F1-score would provide a more meaningful evaluation than overall accuracy alone.

#**Precision** measures the proportion of predicted entities that are actually correct. High precision is important because incorrectly identifying a medical symptom or adverse drug reaction can produce misleading clinical information.

#**Recall** measures the proportion of actual entities that the model successfully identifies. High recall is particularly important for adverse-event detection because missing potentially significant medication side effects could negatively affect pharmacovigilance and patient safety monitoring.

#**F1-score** provides a balanced measure of precision and recall. Macro-averaged F1-score helps evaluate performance across different entity classes without allowing the most frequent class to dominate the results.

#Entity-level precision, recall, and F1-score would be especially valuable because they evaluate whether the model identifies complete AE and SSI spans rather than individual tokens alone. A comprehensive evaluation should report per-class precision, recall, macro-averaged F1, and entity-level F1, while excluding padding positions from meaningful performance calculations.

#For healthcare applications, recall may deserve additional attention when the consequences of missing an adverse drug event are significant. However, precision must also remain sufficiently high to avoid overwhelming clinicians and researchers with false-positive alerts.

### 4. How does the LSTM model compare with the Conditional Random Field (CRF) model?

#The LSTM model differs from the Conditional Random Field (CRF) model used in the previous assignment primarily in how it learns relationships between words and named entities.

#The CRF model relies on manually engineered features, such as word capitalization, neighboring words, prefixes, suffixes, and other linguistic characteristics. This approach offers greater transparency because researchers can examine the features used to make predictions. However, developing effective features requires domain knowledge, experimentation, and additional preparation.

#In contrast, the Bidirectional LSTM uses pretrained GloVe word embeddings and learns contextual representations automatically during training. This reduces the need for extensive manual feature engineering and allows the model to capture more complex patterns in pharmaceutical and clinical language.

#The Bidirectional LSTM can use information from both preceding and following words, which is useful when distinguishing adverse drug reactions from medical conditions. Nevertheless, its internal representations are more difficult to interpret than the manually defined features of a CRF model.

#Although deep learning models can perform well on complex language-processing tasks, their effectiveness depends on training data quality, model architecture, parameter selection, and evaluation methodology. Therefore, neither model should automatically be considered superior without comparing performance on the same evaluation dataset.

### 5. Should deep learning models be explainable, or is selecting the highest-performing model sufficient?

#I believe that explainability is especially important when deploying artificial intelligence models in healthcare, clinical research, and pharmaceutical applications. Although predictive performance is essential, selecting a model based only on accuracy may introduce risks when its predictions influence clinical decisions or medication safety monitoring.

#Healthcare professionals and researchers should be able to understand why a model identifies a particular symptom or adverse drug event. Explainability supports transparency, accountability, error analysis, and trust. It also helps identify possible biases, misleading predictions, and limitations in the training data.

#For example, an NER model analyzing patient medication reviews might incorrectly classify an existing medical condition as an adverse drug reaction. Understanding the contextual evidence behind that prediction could help researchers determine whether the error resulted from ambiguous language, inadequate training examples, or limitations in the model architecture.

#Interpretability tools, such as feature attribution and token-level analysis, may help investigate model behavior, although these techniques do not always provide a complete explanation of the model's reasoning.

#Therefore, I would prioritize a combination of predictive performance, reliability, explainability, fairness, and appropriate human oversight. In healthcare, the highest-performing model is not necessarily the most appropriate model if its limitations cannot be adequately assessed.

### Conclusion

#This assignment demonstrated how Bidirectional LSTM networks and pretrained GloVe embeddings can be applied to Named Entity Recognition in pharmaceutical patient reviews. Increasing the input length can improve contextual understanding and reduce sentence truncation, while additional training epochs may help the model learn more complex relationships.

#My observed validation accuracy of 93.63% demonstrates promising performance, although it remains below the assignment's 95% target. This experience also highlighted the importance of evaluating models beyond overall accuracy, particularly when class imbalance and padding may influence the results.

#For clinical and pharmaceutical applications, precision, recall, macro-averaged F1-score, and entity-level F1-score provide a more comprehensive assessment of whether the model successfully recognizes adverse drug reactions and medical symptoms.

#Ultimately, developing trustworthy healthcare AI requires balancing predictive accuracy, computational efficiency, model interpretability, and patient safety.